# IMPORTS

In [629]:
from pathlib import Path
import warnings

import numpy as np
import pandas as pd
import re

from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, FunctionTransformer

warnings.filterwarnings("ignore")

# setting the seed
RANDOM_STATE = 18
TEST_SIZE = 0.2
DATA_DIR = Path("../data")
TARGET_COL = "House"

pd.set_option("display.max_columns", 200)
pd.set_option("display.width", 120)

# LOAD DATA

In [630]:
characters_path = f"{DATA_DIR}/Characters.csv"
df = pd.read_csv(characters_path, delimiter=';')

print("Shape:", df.shape)
df.head()

Shape: (140, 15)


,Id,Name,Gender,Job,House,Wand,Patronus,Species,Blood status,Hair colour,Eye colour,Loyalty,Skills,Birth,Death
0,1,Harry James Potter,Male,Student,Gryffindor,"11"" Holly phoenix feather",Stag,Human,Half-blood,Black,Bright green,Albus Dumbledore | Dumbledore's Army | Order o...,Parseltongue| Defence Against the Dark Arts | ...,31 July 1980,NaN
1,2,Ronald Bilius Weasley,Male,Student,Gryffindor,"12"" Ash unicorn tail hair",Jack Russell terrier,Human,Pure-blood,Red,Blue,Dumbledore's Army | Order of the Phoenix | Hog...,Wizard chess | Quidditch goalkeeping,1 March 1980,NaN
2,3,Hermione Jean Granger,Female,Student,Gryffindor,"10¾"" vine wood dragon heartstring",Otter,Human,Muggle-born,Brown,Brown,Dumbledore's Army | Order of the Phoenix | Hog...,Almost everything,"19 September, 1979",NaN
3,4,Albus Percival Wulfric Brian Dumbledore,Male,Headmaster,Gryffindor,"15"" Elder Thestral tail hair core",Phoenix,Human,Half-blood,Silver| formerly auburn,Blue,Dumbledore's Army | Order of the Phoenix | Hog...,Considered by many to be one of the most power...,Late August 1881,"30 June, 1997"
4,5,Rubeus Hagrid,Male,Keeper of Keys and Grounds | Professor of Care...,Gryffindor,"16"" Oak unknown core",NaN,Half-Human/Half-Giant,Part-Human (Half-giant),Black,Black,Albus Dumbledore | Order of the Phoenix | Hogw...,Resistant to stunning spells| above average st...,6 December 1928,NaN


In [631]:
df = df.dropna(subset=[TARGET_COL])
print(f"Shape después de eliminar NaN en {TARGET_COL}:", df.shape)

Shape después de eliminar NaN en House: (101, 15)


In [632]:
counts = df[TARGET_COL].value_counts()
df = df[df[TARGET_COL].isin(counts[counts > 1].index)]
print(f"Shape después de eliminar clases poco frecuentes en {TARGET_COL}:", df.shape)

Shape después de eliminar clases poco frecuentes en House: (100, 15)


In [633]:
df.head().to_csv()

',Id,Name,Gender,Job,House,Wand,Patronus,Species,Blood status,Hair colour,Eye colour,Loyalty,Skills,Birth,Death\r\n0,1,Harry James Potter,Male,Student,Gryffindor,"11""  Holly  phoenix feather",Stag,Human,Half-blood,Black,Bright green,Albus Dumbledore | Dumbledore\'s Army | Order of the Phoenix | Hogwarts School of Witchcraft and Wizardry,Parseltongue| Defence Against the Dark Arts | Seeker,31 July 1980,\r\n1,2,Ronald Bilius Weasley,Male,Student,Gryffindor,"12"" Ash unicorn tail hair ",Jack Russell terrier,Human,Pure-blood,Red,Blue,Dumbledore\'s Army | Order of the Phoenix | Hogwarts School of Witchcraft and Wizardry,Wizard chess | Quidditch goalkeeping,1 March 1980,\r\n2,3,Hermione Jean Granger,Female,Student,Gryffindor,"10¾""  vine wood dragon heartstring",Otter,Human,Muggle-born,Brown,Brown,Dumbledore\'s Army | Order of the Phoenix | Hogwarts School of Witchcraft and Wizardry,Almost everything,"19 September,\xa01979",\r\n3,4,Albus Percival Wulfric Brian Dumbledore,Male,Headmaster,Gry

In [634]:
spells_path = f"{DATA_DIR}/Spells.csv"
df_spells = pd.read_csv(spells_path, delimiter=';')

print("Shape:", df_spells.shape)
df_spells.head().to_csv()

Shape: (301, 5)


',Name,Incantation,Type,Effect,Light\r\n0,Summoning Charm,Accio,Charm,Summons an object,\r\n1,Age Line,Unknown,Charm,Prevents people above or below a certain age from access to a target,Blue\r\n2,Water-Making Spell,Aguamenti,"Charm, Conjuration",Conjures\xa0water,Icy blue\r\n3,Launch an object up into the air,Alarte Ascendare,Charm,Rockets target upward,Red\r\n4,Albus Dumbledore\'s Forceful Spell,Unknown,Spell,Great Force,\r\n'

In [635]:
hp1_path = f"{DATA_DIR}/Harry Potter 1.csv"
df_hp1 = pd.read_csv(hp1_path, delimiter=';')

print("Shape:", df_hp1.shape)
df_hp1.head().to_csv()

Shape: (1587, 2)


',Character,Sentence\r\n0,Dumbledore,"I should\'ve known that you would be here, Professor McGonagall."\r\n1,McGonagall,"Good evening, Professor Dumbledore."\r\n2,McGonagall,"Are the rumors true, Albus?"\r\n3,Dumbledore,"I\'m afraid so, professor."\r\n4,Dumbledore,The good and the bad.\r\n'

In [636]:
hp2_path = f"{DATA_DIR}/Harry Potter 2.csv"
df_hp2 = pd.read_csv(hp2_path, delimiter=';')

print("Shape:", df_hp2.shape)
df_hp2.head().to_csv()

Shape: (1700, 2)


',Character,Sentence\r\n0,HARRY ,"I can’t let you out, Hedwig. "\r\n1,HARRY ,I’m not allowed to use magic outside of school. \r\n2,HARRY ,"Besides, if Uncle Vernon…"\r\n3,VERNON,Harry Potter!\r\n4,HARRY,Now you’ve done it.\r\n'

In [637]:
hp3_path = f"{DATA_DIR}/Harry Potter 3.csv"
df_hp3 = pd.read_csv(hp3_path, delimiter=';')

print("Shape:", df_hp3.shape)
df_hp3.head().to_csv()

Shape: (1638, 2)


',CHARACTER,SENTENCE\r\n0,HARRY,Lumos Maxima...\r\n1,HARRY,Lumos Maxima...\r\n2,HARRY,Lumos Maxima...\r\n3,HARRY,Lumos... MAXIMA!\r\n4,AUNT PETUNIA,Harry! Harry!\r\n'

In [638]:
X = df.drop(columns=[TARGET_COL])
y = df[TARGET_COL].astype(str)

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=TEST_SIZE,
    random_state=RANDOM_STATE,
    stratify=y,
)

print("Train shape:", X_train.shape, "Test shape:", X_test.shape)

Train shape: (80, 14) Test shape: (20, 14)


# FEATURE ENGINEERING FUNCTIONS

In [639]:
def normalize_character_name(name: str) -> str:
    if pd.isna(name):
        return 'unknown'
    name = name.lower().strip()
    name = re.sub(r'\s+', ' ', name)
    name = re.sub(r'[^a-z\s]', '', name)
    return name

def load_and_merge_quotes(csv_list):
    dfs = [pd.read_csv(path, delimiter=';') for path in csv_list]
    return pd.concat(dfs, ignore_index=True)

def clean_quotes(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    df['Character'] = df['Character'].apply(normalize_character_name)
    df['Sentence'] = df['Sentence'].astype(str).str.lower().str.strip()
    df['Sentence'] = df['Sentence'].str.replace('"', '', regex=False)
    df = df[df['Sentence'].str.len() > 0]
    return df

def build_quotes_features(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    df['sentence_length'] = df['Sentence'].apply(len)
    df['word_count'] = df['Sentence'].apply(lambda x: len(x.split()))
    df['question'] = df['Sentence'].str.contains(r'\?').astype(int)
    df['exclamation'] = df['Sentence'].str.contains(r'!').astype(int)
    agg = df.groupby('Character').agg({
        'sentence_length': 'mean',
        'word_count': 'mean',
        'question': 'mean',
        'exclamation': 'mean'
    })
    counts = df.groupby('Character').size().to_frame('num_quotes')
    return agg.join(counts).reset_index()

def clean_characters(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    # drop_cols = ['Id', 'Death', 'Birth', 'Loyalty', 'Skills', 'Wand', 'Job']
    # df = df.drop(columns=[c for c in drop_cols if c in df.columns], errors='ignore')
    for col in df.select_dtypes(include='object').columns:
        df[col] = df[col].astype(str).str.lower().str.strip()
    df = df.fillna('unknown')
    return df

def merge_characters_quotes(char_df: pd.DataFrame, quotes_feat: pd.DataFrame):
    df = char_df.copy()
    quotes_feat_copy = quotes_feat.copy()
    merged = df.merge(quotes_feat_copy, left_on='Name', right_on='Character', how='left')
    return merged.fillna(0)

def spells_keyword_features(df: pd.DataFrame, spells_df: pd.DataFrame): # Not using it rn
    df = df.copy()
    spells = spells_df['Incantation'].dropna().str.lower().tolist()
    def count_spells(text):
        return sum(spell in text for spell in spells)
    df['spell_mentions'] = df['Sentence'].apply(count_spells)
    spell_usage = df.groupby('Character')['spell_mentions'].sum().reset_index()
    return spell_usage

In [640]:
quotes_df = load_and_merge_quotes([
    "../data/Harry Potter 1.csv",
    "../data/Harry Potter 2.csv",
    "../data/Harry Potter 3.csv"
])

In [641]:
def normalize_text(text):
    if pd.isna(text):
        return ""
    return re.sub(r'\s+', ' ', str(text).lower().strip())

def replace_character_names_fast(quotes_df, characters_df):

    df = quotes_df.copy()

    char_names = characters_df['Name'].dropna().tolist()
    norm_map = {normalize_text(n): n for n in char_names}

    def match(char):
        char = normalize_text(char)

        # match directo
        if char in norm_map:
            return norm_map[char]

        # fuzzy simple por inclusión
        for k, v in norm_map.items():
            if k in char or char in k:
                return v

        return char

    df['Character'] = df['Character'].apply(match)

    return df
    

In [642]:
# --- Feature engineering y asignación de variables finales ---
# Limpiar y preparar quotes
quotes_clean = clean_quotes(quotes_df)
quotes_clean = replace_character_names_fast(quotes_clean, df)
# Limpiar personajes
X_train_clean = clean_characters(X_train)
X_test_clean = clean_characters(X_test)
# Calcular features de quotes
quotes_features = build_quotes_features(quotes_clean)
# Hacer el merge usando la normalización correcta
X_train_fe = merge_characters_quotes(X_train, quotes_features)
X_test_fe = merge_characters_quotes(X_test, quotes_features)
# Mostrar resultados
print("X_train_fe:")
display(X_train_fe.head(50))
# print("X_test_fe:")
# display(X_test_fe.head())

X_train_fe:


,Id,Name,Gender,Job,Wand,Patronus,Species,Blood status,Hair colour,Eye colour,Loyalty,Skills,Birth,Death,Character,sentence_length,word_count,question,exclamation,num_quotes
0,21,Oliver Wood,Male,Student,Unknown,Unknown,Human,Pure-blood or Half-blood,0,0,Hogwarts School of Witchcraft and Wizardry,Keeper| Captain of the Gryffindor Quidditch team,October 1975 - 31 August 1976,0,Oliver Wood,38.789474,7.026316,0.105263,0.026316,38.0
1,127,Edward Remus Lupin,Male,Student,Unknown,Unknown,Human (Metamorphmagus),Half-blood,Variable,Variable,0,0,"April, 1998",0,0,0.000000,0.000000,0.000000,0.000000,0.0
2,83,Newton Scamander,Male,Employee in the Beast Division of the Departme...,Unknown,Unknown,Human,Pure-blood or half-blood,Red brown,Blue,0,"Magizoology, Order of Merlin, Second Class",24 February 1897,0,Newton Scamander,22.333333,3.000000,0.000000,0.333333,6.0
3,86,Zacharias Smith,Male,Student,Unknown,Unknown,Human,Pure-blood or half-blood,Blonde,0,Dumbledore's Army,Chaser,1 September 1979 - 2 May 1981,0,0,0.000000,0.000000,0.000000,0.000000,0.0
4,31,Molly Weasley,Female,0,Unknown,Unknown,Human,Pure-blood,Red,Bright brown,Order of the Phoenix,Household spells| healing magic,"30 October, 1949",0,0,0.000000,0.000000,0.000000,0.000000,0.0
5,43,Marietta Edgecombe,Female,Student,Unknown,Unknown,Human,Pure-blood or half-blood,Reddish-blonde,Grey,0,Covering up the spots that blight her face aft...,1978-1979,0,0,0.000000,0.000000,0.000000,0.000000,0.0
6,61,Millicent Bulstrode,Female,Student,Unknown,Unknown,Human,Half-blood,Black,0,0,0,1 September 1979- 31 August 1980,0,0,0.000000,0.000000,0.000000,0.000000,0.0
7,65,Penelope Clearwater,Female,Student,Unknown,Non-corporeal,Human,Muggle-born or half-blood[,Blonde,0,0,Prefect,1 September 1975- 31 August 1976,0,Penelope Clearwater,29.000000,4.000000,0.000000,0.000000,1.0
8,105,Fleur Isabelle Delacour,Female,Part-time at Gringotts Wizarding Bank,"9½"", Rosewood, veela hair",Non-corporeal,Human,Quarter-Veela,Silvery-blonde,Dark blue[,Order of the Phoenix,"part Veela, and therefore possesses some of th...","Pre 30 October, 1977",0,0,0.000000,0.000000,0.000000,0.000000,0.0
9,30,Minerva McGonagall,Female,Professor of Transfiguration | Head of Gryffindor,"9½"" Fir dragon heartstring",Cat,Human,Half-blood,Black,Green,Albus Dumbledore | Order of the Phoenix | Hogw...,Animagus,4 October,0,Minerva McGonagall,44.857143,8.090226,0.135338,0.045113,133.0


# STUFF ABOUT MAGIC

In [643]:
def clean_spells_for_features(spells_df):
    df = spells_df.copy()

    df['Incantation'] = (
        df['Incantation']
        .fillna('')
        .astype(str)
        .str.lower()
        .str.strip()
    )

    df['Type'] = (
        df['Type']
        .fillna('unknown')
        .astype(str)
        .str.lower()
        .str.strip()
    )

    # eliminar vacíos o unknown
    df = df[df['Incantation'] != '']
    df = df[df['Incantation'] != 'unknown']

    return df

In [644]:
quotes_df['Sentence'] = (
    quotes_df['Sentence']
    .fillna('')
    .astype(str)
    .str.lower()
)

In [645]:
def build_spell_type_features(quotes_df, spells_df):

    quotes = quotes_df.copy()
    spells = clean_spells_for_features(spells_df)

    spell_dict = dict(zip(spells['Incantation'], spells['Type']))

    def detect_spell_type(sentence):
        sentence = str(sentence)

        found_types = [
            typ for spell, typ in spell_dict.items()
            if spell and spell in sentence
        ]

        return found_types if found_types else ['none']

    quotes['spell_types'] = quotes['Sentence'].apply(detect_spell_type)

    quotes = quotes.reset_index(drop=True)

    exploded = quotes.explode('spell_types')
    exploded = exploded.dropna(subset=['Character', 'spell_types'])

    features = (
        exploded
        .groupby(['Character', 'spell_types'])
        .size()
        .unstack(fill_value=0)
        .reset_index()
    )

    return features

- Gryffindor → más “Charm”
- Slytherin → más “Curse”
- Ravenclaw → más variedad / complejidad
- Hufflepuff → menos agresividad

In [646]:
# Intensidad mágica

def spell_intensity_features(quotes_df, spells_df):
    quotes = quotes_df.copy()
    spells = clean_spells_for_features(spells_df)
    spells_list = spells['Incantation'].dropna().str.lower().tolist()

    def count_spells(sentence):
        sentence = str(sentence).lower()  # Ensure sentence is string
        return sum(spell in sentence for spell in spells_list if spell != 'unknown')

    quotes['spell_count'] = quotes['Sentence'].apply(count_spells)

    features = quotes.groupby('Character')['spell_count'].agg([
        'sum', 'mean', 'max'
    ]).reset_index()

    features.columns = ['Character', 'spell_total', 'spell_avg', 'spell_max']

    return features

- Personajes más activos mágicamente
- Uso frecuente vs ocasional

In [647]:
# Magia oscura
def dark_magic_features(quotes_df, spells_df):
    quotes = quotes_df.copy()
    spells = clean_spells_for_features(spells_df)

    spells['Type'] = spells['Type'].str.lower()

    dark_spells = spells[
        spells['Type'].str.contains('curse|dark', na=False)
    ]['Incantation'].dropna().str.lower().tolist()

    def is_dark(sentence):
        sentence = str(sentence).lower()  # Ensure sentence is string
        return any(spell in sentence for spell in dark_spells)

    quotes['dark_magic'] = quotes['Sentence'].apply(is_dark).astype(int)

    features = quotes.groupby('Character')['dark_magic'].mean().reset_index()

    return features

- Tendencia a magia oscura → Slytherin predictor fuerte

In [648]:
# Diversidad de hechizos
def spell_diversity_features(quotes_df, spells_df):
    quotes = quotes_df.copy()
    spells = clean_spells_for_features(spells_df)
    spells_list = spells['Incantation'].dropna().str.lower().tolist()

    def extract_spells(sentence):
        sentence = str(sentence).lower()  # Ensure sentence is string
        return [spell for spell in spells_list if spell in sentence]

    quotes['spells_used'] = quotes['Sentence'].apply(extract_spells)

    exploded = quotes.explode('spells_used')

    diversity = exploded.groupby('Character')['spells_used'].nunique().reset_index()
    diversity.columns = ['Character', 'unique_spells_used']

    return diversity

- Ravenclaw → alta diversidad
- Gryffindor → uso más directo/repetitivo

In [649]:
# UNIFICAR ALL
def build_all_spell_features(quotes_df, spells_df):
    f1 = build_spell_type_features(quotes_df, spells_df)
    f2 = spell_intensity_features(quotes_df, spells_df)
    f3 = dark_magic_features(quotes_df, spells_df)
    f4 = spell_diversity_features(quotes_df, spells_df)

    # NO normalices aquí, deja los nombres como están
    df = f1.merge(f2, on='Character', how='outer')
    df = df.merge(f3, on='Character', how='outer')
    df = df.merge(f4, on='Character', how='outer')

    return df.fillna(0)

In [650]:
spell_features = build_all_spell_features(quotes_clean, df_spells)

# Merge usando el nombre original
X_train_fe = X_train_fe.merge(spell_features, left_on='Name', right_on='Character', how='left')
X_test_fe = X_test_fe.merge(spell_features, left_on='Name', right_on='Character', how='left')

X_train_fe = X_train_fe.fillna(0)
X_test_fe = X_test_fe.fillna(0)

print("X_train_fe:")
display(X_train_fe.head())
print("X_test_fe:")
display(X_test_fe.head())

X_train_fe:


,Id,Name,Gender,Job,Wand,Patronus,Species,Blood status,Hair colour,Eye colour,Loyalty,Skills,Birth,Death,Character_x,sentence_length,word_count,question,exclamation,num_quotes,Character_y,charm,conjuration,counter-spell,curse,"healing spell, charm",none,spell,transfiguration,vanishment,spell_total,spell_avg,spell_max,dark_magic,unique_spells_used
0,21,Oliver Wood,Male,Student,Unknown,Unknown,Human,Pure-blood or Half-blood,0,0,Hogwarts School of Witchcraft and Wizardry,Keeper| Captain of the Gryffindor Quidditch team,October 1975 - 31 August 1976,0,Oliver Wood,38.789474,7.026316,0.105263,0.026316,38.0,Oliver Wood,0.0,0.0,0.0,0.0,0.0,38.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,127,Edward Remus Lupin,Male,Student,Unknown,Unknown,Human (Metamorphmagus),Half-blood,Variable,Variable,0,0,"April, 1998",0,0,0.000000,0.000000,0.000000,0.000000,0.0,0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,83,Newton Scamander,Male,Employee in the Beast Division of the Departme...,Unknown,Unknown,Human,Pure-blood or half-blood,Red brown,Blue,0,"Magizoology, Order of Merlin, Second Class",24 February 1897,0,Newton Scamander,22.333333,3.000000,0.000000,0.333333,6.0,Newton Scamander,0.0,0.0,0.0,0.0,0.0,6.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,86,Zacharias Smith,Male,Student,Unknown,Unknown,Human,Pure-blood or half-blood,Blonde,0,Dumbledore's Army,Chaser,1 September 1979 - 2 May 1981,0,0,0.000000,0.000000,0.000000,0.000000,0.0,0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,31,Molly Weasley,Female,0,Unknown,Unknown,Human,Pure-blood,Red,Bright brown,Order of the Phoenix,Household spells| healing magic,"30 October, 1949",0,0,0.000000,0.000000,0.000000,0.000000,0.0,0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


X_test_fe:


,Id,Name,Gender,Job,Wand,Patronus,Species,Blood status,Hair colour,Eye colour,Loyalty,Skills,Birth,Death,Character_x,sentence_length,word_count,question,exclamation,num_quotes,Character_y,charm,conjuration,counter-spell,curse,"healing spell, charm",none,spell,transfiguration,vanishment,spell_total,spell_avg,spell_max,dark_magic,unique_spells_used
0,37,Filius Flitwick,Male,Professor of Charms | Head of Ravenclaw,Unknown,Non-corporeal,Human(goblin ancestry),Part-Goblin,White,0,0,Charms,17 October 1958 or earlier,0,Filius Flitwick,28.153846,4.846154,0.153846,0.384615,13.0,Filius Flitwick,1.0,0.0,0.0,0.0,0.0,12.0,0.0,0.0,0.0,1.0,0.076923,1.0,0.0,1.0
1,25,Lavender Brown,Female,Student,Unknown,Unknown,Human,Pure-blood,Blond,Blue,Dumbledore's Army |Hogwarts School of Witchcra...,Divination,1 September 1979- 31 August 1980,"2 May, 1998",0,0.000000,0.000000,0.000000,0.000000,0.0,0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,0.0,0.0,0.0
2,16,Peter Pettigrew,Male,The Servant of Lord Voldemort,"9¼"" Chestnut dragon heartstring",0,Human,Half-blood or pure-blood,Colourless and balding,Blue,Lord Voldemort| Death Eaters,Animagus,1 September 1959- 31 August 1960,Late March 1998,0,0.000000,0.000000,0.000000,0.000000,0.0,0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,0.0,0.0,0.0
3,57,Blaise Zabini,Male,Student,Unknown,Unknown,Human,Pure-blood or half-blood,Black,Brown,0,Chaser,"1 September, 1979 – 21 April, 1980",0,0,0.000000,0.000000,0.000000,0.000000,0.0,0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,0.0,0.0,0.0
4,68,Salazar Slytherin,Male,Founder of Slytherin,"snakewood, Basilisk horn",Unknown,Human,Pure-blood,Grey,Dark,0,Accomplished Legilimens and one of the first r...,Pre 976,11th century,0,0.000000,0.000000,0.000000,0.000000,0.0,0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,0.0,0.0,0.0


# FEATURE SELECTION

In [651]:
drop_cols = ['char_key','Id', 'Death', 'Birth', 'Loyalty', 'Skills', 'Wand', 'Job']
X_train_fe = X_train_fe.drop(columns=drop_cols, errors='ignore')
X_test_fe = X_test_fe.drop(columns=drop_cols, errors='ignore')

print("X_train_fe:")
display(X_train_fe.head(50))
print("X_test_fe:")
display(X_test_fe.head(20))

X_train_fe:


,Name,Gender,Patronus,Species,Blood status,Hair colour,Eye colour,Character_x,sentence_length,word_count,question,exclamation,num_quotes,Character_y,charm,conjuration,counter-spell,curse,"healing spell, charm",none,spell,transfiguration,vanishment,spell_total,spell_avg,spell_max,dark_magic,unique_spells_used
0,Oliver Wood,Male,Unknown,Human,Pure-blood or Half-blood,0,0,Oliver Wood,38.789474,7.026316,0.105263,0.026316,38.0,Oliver Wood,0.0,0.0,0.0,0.0,0.0,38.0,0.0,0.0,0.0,0.0,0.000000,0.0,0.0,0.0
1,Edward Remus Lupin,Male,Unknown,Human (Metamorphmagus),Half-blood,Variable,Variable,0,0.000000,0.000000,0.000000,0.000000,0.0,0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,0.0,0.0,0.0
2,Newton Scamander,Male,Unknown,Human,Pure-blood or half-blood,Red brown,Blue,Newton Scamander,22.333333,3.000000,0.000000,0.333333,6.0,Newton Scamander,0.0,0.0,0.0,0.0,0.0,6.0,0.0,0.0,0.0,0.0,0.000000,0.0,0.0,0.0
3,Zacharias Smith,Male,Unknown,Human,Pure-blood or half-blood,Blonde,0,0,0.000000,0.000000,0.000000,0.000000,0.0,0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,0.0,0.0,0.0
4,Molly Weasley,Female,Unknown,Human,Pure-blood,Red,Bright brown,0,0.000000,0.000000,0.000000,0.000000,0.0,0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,0.0,0.0,0.0
5,Marietta Edgecombe,Female,Unknown,Human,Pure-blood or half-blood,Reddish-blonde,Grey,0,0.000000,0.000000,0.000000,0.000000,0.0,0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,0.0,0.0,0.0
6,Millicent Bulstrode,Female,Unknown,Human,Half-blood,Black,0,0,0.000000,0.000000,0.000000,0.000000,0.0,0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,0.0,0.0,0.0
7,Penelope Clearwater,Female,Non-corporeal,Human,Muggle-born or half-blood[,Blonde,0,Penelope Clearwater,29.000000,4.000000,0.000000,0.000000,1.0,Penelope Clearwater,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.000000,0.0,0.0,0.0
8,Fleur Isabelle Delacour,Female,Non-corporeal,Human,Quarter-Veela,Silvery-blonde,Dark blue[,0,0.000000,0.000000,0.000000,0.000000,0.0,0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,0.0,0.0,0.0
9,Minerva McGonagall,Female,Cat,Human,Half-blood,Black,Green,Minerva McGonagall,44.857143,8.090226,0.135338,0.045113,133.0,Minerva McGonagall,0.0,0.0,0.0,0.0,0.0,131.0,0.0,2.0,0.0,2.0,0.015038,1.0,0.0,1.0


X_test_fe:


,Name,Gender,Patronus,Species,Blood status,Hair colour,Eye colour,Character_x,sentence_length,word_count,question,exclamation,num_quotes,Character_y,charm,conjuration,counter-spell,curse,"healing spell, charm",none,spell,transfiguration,vanishment,spell_total,spell_avg,spell_max,dark_magic,unique_spells_used
0,Filius Flitwick,Male,Non-corporeal,Human(goblin ancestry),Part-Goblin,White,0,Filius Flitwick,28.153846,4.846154,0.153846,0.384615,13.0,Filius Flitwick,1.0,0.0,0.0,0.0,0.0,12.0,0.0,0.0,0.0,1.0,0.076923,1.0,0.000000,1.0
1,Lavender Brown,Female,Unknown,Human,Pure-blood,Blond,Blue,0,0.000000,0.000000,0.000000,0.000000,0.0,0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,0.0,0.000000,0.0
2,Peter Pettigrew,Male,0,Human,Half-blood or pure-blood,Colourless and balding,Blue,0,0.000000,0.000000,0.000000,0.000000,0.0,0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,0.0,0.000000,0.0
3,Blaise Zabini,Male,Unknown,Human,Pure-blood or half-blood,Black,Brown,0,0.000000,0.000000,0.000000,0.000000,0.0,0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,0.0,0.000000,0.0
4,Salazar Slytherin,Male,Unknown,Human,Pure-blood,Grey,Dark,0,0.000000,0.000000,0.000000,0.000000,0.0,0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,0.0,0.000000,0.0
5,Neville Longbottom,Male,Non-corporeal,Human,Pure-blood,Blond,0,Neville Longbottom,28.958333,5.375000,0.208333,0.416667,24.0,Neville Longbottom,0.0,0.0,0.0,0.0,0.0,24.0,0.0,0.0,0.0,0.0,0.000000,0.0,0.000000,0.0
6,Ginevra (Ginny) Molly Weasley,Female,Horse,Human,Pure-blood,Red,Bright brown,Ginevra (Ginny) Molly Weasley,21.428571,4.285714,0.142857,0.000000,7.0,Ginevra (Ginny) Molly Weasley,0.0,0.0,0.0,0.0,0.0,7.0,0.0,0.0,0.0,0.0,0.000000,0.0,0.000000,0.0
7,Gregory Goyle,Male,Unknown,Human,Pure-blood,Brown,0,0,0.000000,0.000000,0.000000,0.000000,0.0,0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,0.0,0.000000,0.0
8,Rodolphus Lestrange,Male,Unknown,Human,Pure-blood,Dark,Dark,0,0.000000,0.000000,0.000000,0.000000,0.0,0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,0.0,0.000000,0.0
9,Lucius Malfoy,Male,Unknown,Human,Pure-blood,White-blond,Grey,Lucius Malfoy,33.973333,6.453333,0.186667,0.026667,75.0,Lucius Malfoy,0.0,0.0,0.0,0.0,0.0,75.0,0.0,0.0,0.0,0.0,0.000000,0.0,0.000000,0.0
